Imagine you are building a tool for an accounting department. People just want to type or paste messy, unstructured text about an expense, and your local AI needs to instantly spit out clean, validated JSON ready for a database.

In [1]:
import instructor
from pydantic import BaseModel, Field
from openai import OpenAI
from typing import Optional

# 1. Define the exact structure we need to extract
    # Field descriptions—this acts as micro-prompting to give the 8B model explicit instructions for each exact piece of data.
class ExpenseRecord(BaseModel):
    name: str = Field(description="The name of the person who made the purchase")
    date: str = Field(description="The date of the transaction. Format as YYYY-MM-DD if possible.")
    amount: float = Field(description="The numeric cost of the transaction")
    item: Optional[str] = Field(description="A brief description of what was purchased")

# 2. Set up our disguised Instructor client
client = instructor.from_openai(
    OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama", 
    ),
    mode=instructor.Mode.JSON,
)

print("🧾 CLI Expense Extractor Started! Type 'exit' to quit.\n" + "-"*45)

# 3. The Continuous Loop
while True:
    raw_text = input("\nEnter messy receipt text: ")
    
    if raw_text.lower() == 'exit':
        print("Shutting down extractor. Goodbye!")
        break
        
    print("🤖 Processing...", end="\r") # The \r overwrites this line when done
    
    try:
        # 4. The Extraction Magic
            # client (The Messenger): This is the object we set up earlier in the script.
            # .chat: By adding .chat, you are telling the client: "Route this request to the conversational text department." (Like the FastAPI setup)
            # .create(...): This is the actual action verb. Up until this word, we were just pointing to a location in the API. 
                # create() is the function that actually fires the HTTP request over your local network to the Ollama engine.
        extracted_data = client.chat.completions.create(
            model="llama3.1",
            response_model=ExpenseRecord,
            messages=[
                {
                    "role": "system", 
                    "content": "You are a world-class data extraction algorithm. Extract the requested fields from the user's input."
                },
                {"role": "user", "content": raw_text}
            ]
        )
        
        # 5. Print the clean data
        print("✅ Extraction Complete!             ")
        print(extracted_data.model_dump_json(indent=2))
        
    except Exception as e:
         # In case the model completely fails or the connection drops
        print(f"❌ An error occurred: {e}")

🧾 CLI Expense Extractor Started! Type 'exit' to quit.
---------------------------------------------
✅ Extraction Complete!             
{
  "name": "",
  "date": "",
  "amount": 40.0,
  "item": "coffee"
}
Shutting down extractor. Goodbye!
